# Observatorio del Arbolado Urbano de Temuco
## Notebook de análisis satelital — Google Earth Engine

Cuantifica la **pérdida de cobertura arbórea (dosel)** en Temuco entre 2005 y 2025 usando
Landsat y Sentinel-2, con NDVI, altura de dosel (ETH 10 m), detección de cambio y temperatura
superficial (LST). El procesamiento ocurre en la nube de Google Earth Engine (GEE).

**Modo de uso:**
1. Necesitas una cuenta de **Google Earth Engine** aprobada (registro gratuito para investigación en https://earthengine.google.com).
2. Ejecuta las celdas de arriba hacia abajo.
3. Por defecto corre en `TEST_MODE = True`: solo 2005 vs 2025 sobre un sector pequeño (rápido). 
   Para el análisis completo de los 5 cortes sobre toda la ciudad, pon `TEST_MODE = False` en la celda de configuración.

> Método alineado con `PLAN_TECNICO_SATELITAL.md` (§12 mejoras: HLS/homogeneización, altura de dosel, CCDC, Fmask, i-Tree/LST).

## 0. Instalación de dependencias
Solo la primera vez (en Google Colab ya viene `earthengine-api`; `geemap` se instala rápido).

In [ ]:
# Ejecutar una vez. En Colab descomenta la línea de abajo.
# %pip install earthengine-api geemap

## 1. Autenticación e inicialización
La primera vez abrirá una ventana para autorizar tu cuenta Google. Reemplaza `TU_PROYECTO_GEE`
por el ID de tu proyecto de Cloud asociado a Earth Engine (aparece al registrarte).

In [ ]:
import ee, geemap

# ID del proyecto de Earth Engine (ya configurado):
EE_PROJECT = 'arbolado-urbano-502605'

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print('Earth Engine inicializado:', ee.String('OK').getInfo())

## 2. Configuración
Toda la parametrización del análisis vive aquí. Cambia `TEST_MODE` para pasar de la prueba
rápida al análisis completo.

- **Serie A (por defecto):** Landsat 30 m en todos los cortes → comparabilidad (recomendado para la métrica oficial).
- La ventana de verano austral (dic–mar) capta el dosel pleno y minimiza nubes.

In [ ]:
# ------------------- INTERRUPTOR PRINCIPAL -------------------
TEST_MODE = True   # True = 2005 vs 2025 en un sector chico (rápido). False = 5 cortes, ciudad completa.

# ------------------- Área de interés (AOI) -------------------
# Bounding box aproximado del área urbana de Temuco (lon_min, lat_min, lon_max, lat_max).
TEMUCO_URBANO = ee.Geometry.Rectangle([-72.685, -38.790, -72.530, -38.680])

# Sector de prueba (zona centro / eje Av. Imperial - Caupolicán):
SECTOR_TEST   = ee.Geometry.Rectangle([-72.620, -38.755, -72.575, -38.725])

# LÍMITE URBANO OFICIAL (recomendado para producción). GeoJSON ya descargado del IDE Municipal
# de Temuco en investigacion/geodata/. Pon True para usarlo en vez del bounding box.
USE_OFFICIAL_BOUNDARY  = False
OFFICIAL_BOUNDARY_PATH = 'investigacion/geodata/limite_urbano_temuco.geojson'

AOI = SECTOR_TEST if TEST_MODE else TEMUCO_URBANO
if USE_OFFICIAL_BOUNDARY and not TEST_MODE:
    import json
    with open(OFFICIAL_BOUNDARY_PATH, encoding='utf-8') as _f:
        _gj = json.load(_f)
    AOI = geemap.geojson_to_ee(_gj).geometry()
    print('AOI = límite urbano OFICIAL de Temuco (IDE Municipal).')

# ------------------- Cortes temporales -------------------
YEARS = [2005, 2025] if TEST_MODE else [2005, 2010, 2015, 2020, 2025]

# Ventana de verano austral: 01-dic (año-1) a 15-mar (año)
SEASON_START = '-12-01'   # se aplica al año anterior
SEASON_END   = '-03-15'   # se aplica al año del corte

# ------------------- Umbrales de dosel -------------------
NDVI_THRESHOLD   = 0.60    # NDVI mínimo candidato a vegetación densa
CANOPY_HEIGHT_M  = 3.0     # altura mínima (m) para considerar ÁRBOL (excluye pasto/césped)
USE_CANOPY_HEIGHT = True   # cruzar NDVI con altura de dosel ETH (solo válido ~2020)

MAX_CLOUD = 60             # % de nube máximo por escena antes de compositar
SCALE_M = 30               # resolución de trabajo (Serie A Landsat)

print('TEST_MODE =', TEST_MODE, '| Años:', YEARS, '| AOI:', 'sector' if TEST_MODE else 'ciudad')

## 3. Sensores y funciones de preprocesamiento
Selección de sensor por año (Serie A = Landsat), enmascarado de nubes/sombra con la banda de calidad
`QA_PIXEL` (equivalente Fmask en Collection 2), escalado a reflectancia y cálculo de NDVI.

In [ ]:
# Colección Landsat Collection 2, Nivel 2 (reflectancia superficial + banda térmica ST)
def sensor_for_year(year):
    if year <= 2011:
        return 'LT05'   # Landsat 5 TM
    elif year <= 2021:
        return 'LC08'   # Landsat 8 OLI
    else:
        return 'LC09'   # Landsat 9 OLI-2

COLLECTIONS = {
    'LT05': 'LANDSAT/LT05/C02/T1_L2',
    'LC08': 'LANDSAT/LC08/C02/T1_L2',
    'LC09': 'LANDSAT/LC09/C02/T1_L2',
}

# Bandas NIR/Rojo y térmica según sensor
BANDS = {
    'LT05': {'nir': 'SR_B4', 'red': 'SR_B3', 'thermal': 'ST_B6'},
    'LC08': {'nir': 'SR_B5', 'red': 'SR_B4', 'thermal': 'ST_B10'},
    'LC09': {'nir': 'SR_B5', 'red': 'SR_B4', 'thermal': 'ST_B10'},
}

def mask_and_scale(img):
    '''Enmascara nube dilatada (bit 1), nube (bit 3) y sombra de nube (bit 4); escala reflectancia.'''
    qa = img.select('QA_PIXEL')
    mask = (qa.bitwiseAnd(1 << 1).eq(0)
              .And(qa.bitwiseAnd(1 << 3).eq(0))
              .And(qa.bitwiseAnd(1 << 4).eq(0)))
    optical = img.select('SR_B.*').multiply(0.0000275).add(-0.2)
    return img.addBands(optical, None, True).updateMask(mask)

def ndvi_composite(year, aoi):
    '''Compuesto mediana de NDVI para la ventana de verano del año dado.'''
    s = sensor_for_year(year)
    start = ee.Date(str(year - 1) + SEASON_START)
    end   = ee.Date(str(year) + SEASON_END)
    col = (ee.ImageCollection(COLLECTIONS[s])
           .filterBounds(aoi)
           .filterDate(start, end)
           .filter(ee.Filter.lt('CLOUD_COVER', MAX_CLOUD))
           .map(mask_and_scale))
    b = BANDS[s]
    ndvi = col.map(lambda im: im.normalizedDifference([b['nir'], b['red']]).rename('NDVI'))
    n = col.size()
    return ndvi.median().rename('NDVI').clip(aoi), n

print('Funciones de sensor listas.')

## 4. Compuestos NDVI por año
Construye un compuesto de NDVI limpio (sin nubes) por cada corte temporal.

In [ ]:
ndvi_by_year = {}
for y in YEARS:
    img, n = ndvi_composite(y, AOI)
    ndvi_by_year[y] = img
    print(f'{y}: sensor {sensor_for_year(y)} — escenas usadas: {n.getInfo()}')

## 5. Altura de dosel: separar ÁRBOL de PASTO
El NDVI alto no distingue un árbol de un césped. Cruzamos NDVI con el **ETH Global Canopy Height
10 m (Lang et al. 2023)** para exigir altura > 3 m. Nota: la capa de altura es del año ~2020, así
que es una *máscara estructural* de referencia (no varía año a año); para años sin LiDAR se usa como
aproximación de qué píxeles corresponden a arbolado.

In [ ]:
# ETH Global Canopy Height 2020 (10 m). Si el asset no está disponible en tu cuenta,
# alternativa: capa de la comunidad 'projects/sat-io/open-datasets/GLOBAL-CANOPY-HEIGHT-2020'.
try:
    canopy_height = ee.Image('users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1').rename('height')
    _ = canopy_height.bandNames().getInfo()
    HEIGHT_OK = True
except Exception as e:
    print('Aviso: no se pudo cargar ETH canopy height (', e, ') -> se usará solo NDVI.')
    HEIGHT_OK = False

def canopy_mask(ndvi_img):
    '''Máscara binaria de dosel arbóreo: NDVI alto (+ altura > umbral si está disponible).'''
    m = ndvi_img.gt(NDVI_THRESHOLD)
    if USE_CANOPY_HEIGHT and HEIGHT_OK:
        m = m.And(canopy_height.gt(CANOPY_HEIGHT_M))
    return m.rename('canopy').selfMask()

canopy_by_year = {y: canopy_mask(ndvi_by_year[y]) for y in YEARS}
print('Máscaras de dosel calculadas. Altura de dosel:', 'SÍ' if (USE_CANOPY_HEIGHT and HEIGHT_OK) else 'no')

## 6. Mapa comparativo (antes / después)
Visor interactivo con deslizador. Verde intenso = dosel; ocre/gris = suelo/cemento.

In [ ]:
y0, y1 = YEARS[0], YEARS[-1]
ndvi_vis = {'min': 0.0, 'max': 0.85, 'palette': ['#b0997a', '#d9ca9a', '#9cbf6a', '#4a7c3f', '#123a29']}

Map = geemap.Map(center=[-38.74, -72.60], zoom=13)
left  = geemap.ee_tile_layer(ndvi_by_year[y0], ndvi_vis, f'NDVI {y0}')
right = geemap.ee_tile_layer(ndvi_by_year[y1], ndvi_vis, f'NDVI {y1}')
Map.split_map(left_layer=left, right_layer=right)
Map.addLayer(AOI, {'color': 'red'}, 'AOI', True, 0.3)
Map

## 7. Detección de cambio: ΔNDVI (mapa de calor de pérdida)
Resta el NDVI final menos el inicial. Rojo = pérdida de verde; verde = ganancia/estable.

In [ ]:
delta = ndvi_by_year[y1].subtract(ndvi_by_year[y0]).rename('dNDVI')
delta_vis = {'min': -0.4, 'max': 0.4, 'palette': ['#8b1a1a', '#c85a2b', '#f2f2f2', '#8fb56a', '#123a29']}

Map2 = geemap.Map(center=[-38.74, -72.60], zoom=13)
Map2.addLayer(delta, delta_vis, f'ΔNDVI {y0}→{y1}')
Map2.add_colorbar(delta_vis, label=f'ΔNDVI ({y0}→{y1})  rojo=pérdida')
Map2

## 8. Cuantificación: hectáreas de dosel por año y pérdida neta
Suma el área de los píxeles clasificados como dosel en cada corte.

In [ ]:
def canopy_hectares(mask_img, aoi):
    area = mask_img.multiply(ee.Image.pixelArea())
    stat = area.reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi, scale=SCALE_M, maxPixels=1e11)
    m2 = ee.Number(stat.get('canopy'))
    return m2.divide(1e4)  # hectáreas

resultados = {}
for y in YEARS:
    ha = canopy_hectares(canopy_by_year[y], AOI).getInfo()
    resultados[y] = ha
    print(f'{y}: {ha:,.1f} ha de dosel')

ha0, ha1 = resultados[y0], resultados[y1]
perdida_ha = ha0 - ha1
perdida_pct = (perdida_ha / ha0 * 100) if ha0 else 0
print('\n--- RESUMEN ---')
print(f'Dosel {y0}: {ha0:,.1f} ha')
print(f'Dosel {y1}: {ha1:,.1f} ha')
print(f'Pérdida neta {y0}-{y1}: {perdida_ha:,.1f} ha  ({perdida_pct:,.1f} %)')

## 9. Tabla de resultados (pandas)
Exportable a CSV/Excel para el informe.

In [ ]:
import pandas as pd
df = pd.DataFrame({'anio': list(resultados.keys()),
                   'dosel_ha': [round(v, 1) for v in resultados.values()]})
df['perdida_ha_vs_inicio'] = round(df['dosel_ha'].iloc[0], 1) - df['dosel_ha']
df['perdida_pct_vs_inicio'] = (df['perdida_ha_vs_inicio'] / df['dosel_ha'].iloc[0] * 100).round(1)
df
# df.to_csv('dosel_temuco.csv', index=False)

## 10. Servicios ecosistémicos: temperatura superficial (LST)
Extrae la temperatura de superficie (°C) del compuesto de verano y permite comparar barrios que
perdieron dosel con los que lo conservan (isla de calor).

In [ ]:
def lst_celsius(year, aoi):
    s = sensor_for_year(year)
    start = ee.Date(str(year - 1) + SEASON_START)
    end   = ee.Date(str(year) + SEASON_END)
    col = (ee.ImageCollection(COLLECTIONS[s])
           .filterBounds(aoi).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUD_COVER', MAX_CLOUD))
           .map(mask_and_scale))
    tb = BANDS[s]['thermal']
    def to_c(img):
        return img.select(tb).multiply(0.00341802).add(149.0).subtract(273.15).rename('LST')
    return col.map(to_c).median().clip(aoi)

lst = lst_celsius(y1, AOI)
lst_vis = {'min': 15, 'max': 40, 'palette': ['#2b6cb0', '#7fc97f', '#f6e58d', '#e08a54', '#8b1a1a']}
Map3 = geemap.Map(center=[-38.74, -72.60], zoom=13)
Map3.addLayer(lst, lst_vis, f'LST °C {y1}')
Map3.add_colorbar(lst_vis, label=f'Temperatura superficial °C ({y1})')
Map3

## 11. Exportar mapas (opcional)
Descomenta para exportar los rásteres a tu Google Drive como GeoTIFF.

In [ ]:
# task = ee.batch.Export.image.toDrive(
#     image=delta, description=f'dNDVI_{y0}_{y1}', folder='ObservatorioArbolado',
#     region=AOI, scale=SCALE_M, maxPixels=1e11)
# task.start()
# print('Exportación iniciada; revisa la pestaña Tasks en code.earthengine.google.com')

## 12. Próximos pasos y notas

- **Escalar:** pon `TEST_MODE = False` para correr los 5 cortes sobre toda la ciudad, y
  `USE_OFFICIAL_BOUNDARY = True` para usar el **límite urbano oficial** ya descargado en
  `investigacion/geodata/limite_urbano_temuco.geojson` (IDE Municipal de Temuco).
- **Validación con capa oficial:** `investigacion/geodata/areas_verdes_temuco.geojson` (1.150
  áreas verdes del PRC) sirve para contrastar el dosel detectado y calcular m²/hab por sector.
- **Detección de cambio robusta:** este notebook usa ΔNDVI de 2 fechas (comunicacional). Para la
  serie completa conviene migrar a **CCDC** o **LandTrendr** (disponibles en GEE) — ver §12 del plan técnico.
- **Validación ciudadana:** las mediciones de terreno (copa, GPS) del Observatorio sirven como
  puntos de entrenamiento/validación para una clasificación supervisada (Random Forest).
- **Valoración económica:** cruzar los polígonos de pérdida con especie/ubicación y aplicar la
  fórmula del **Art. 32° de la Ordenanza 004/2021** (`Vt = Vb·fd·fu·fs`) para estimar el costo en UTM.
- **Limitación:** a 30 m Landsat mide masas de dosel, no árboles individuales; la escala ciudadana
  aporta el detalle fino.